# 🧹 Phase 2 : Data Wrangling (Nettoyage et Préparation)

Dans ce notebook, nous allons préparer notre jeu de données brut pour l'analyse. <br>
**Objectif :** Nettoyer les données (gestion des valeurs manquantes, typage correct des colonnes) et les transformer dans un format exploitable par nos futurs algorithmes.<br>

Nous commençons par importer les librairies et charger les données brutes que nous avons acquises dans la phase précédente.

### 1. Initialisation et imports

In [15]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
# Ajout du dossier parent au chemin de recherche des modules
sys.path.append(os.path.abspath('..'))
import plotly.graph_objects as go
from plotly.subplots import make_subplots
print("Librairies prêtes pour le Wrangling !")

Librairies prêtes pour le Wrangling !


### 2. Chargement du dataset brut et Audit Initial

Chargement du dataset **Automobile** 

In [2]:
raw_data_path = '../data/raw/Automobile.csv'
df_raw = pd.read_csv(raw_data_path)

Avant de nettoyer, nous devons observer l'état actuel de nos données brutes. <br>
L'objectif de la cellule suivante est de faire un état des lieux complet : vérifier la taille du dataset, repérer les types de colonnes , compter les valeurs manquantes et s'assurer de l'absence de doublons.

In [3]:
print("=== Dimensions ===")
print(f"Shape : {df_raw.shape}")

print("\n=== Types des colonnes ===")
print(df_raw.dtypes)

print("\n=== Valeurs manquantes par colonne ===")
print(df_raw.isnull().sum())

print("\n=== Nombre de doublons ===")
print(df_raw.duplicated().sum())

print("\n=== Statistiques descriptives ===")
df_raw.describe()

=== Dimensions ===
Shape : (398, 9)

=== Types des colonnes ===
name                str
mpg             float64
cylinders         int64
displacement    float64
horsepower      float64
weight            int64
acceleration    float64
model_year        int64
origin              str
dtype: object

=== Valeurs manquantes par colonne ===
name            0
mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
dtype: int64

=== Nombre de doublons ===
0

=== Statistiques descriptives ===


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year
count,398.000000,398.000000,398.000000,392.000000,398.000000,398.000000,398.000000
mean,23.514573,5.454774,193.425879,104.469388,2970.424623,15.568090,76.010050
std,7.815984,1.701004,104.269838,38.491160,846.841774,2.757689,3.697627
min,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,70.000000
25%,17.500000,4.000000,104.250000,75.000000,2223.750000,13.825000,73.000000
50%,23.000000,4.000000,148.500000,93.500000,2803.500000,15.500000,76.000000
75%,29.000000,8.000000,262.000000,126.000000,3608.000000,17.175000,79.000000
max,46.600000,8.000000,455.000000,230.000000,5140.000000,24.800000,82.000000


**Interprétation de l'état des lieux :** L'inspection des `dtypes` révèle un problème classique du dataset *Automobile* : des colonnes qui devraient être purement numériques (comme `price`, `horsepower`, ou `peak-rpm`) sont identifiées avec le type `object` (texte). 

Comme expliqué dans le cours, cela signifie qu'il y a des caractères non numériques (probablement des symboles pour indiquer des données manquantes) cachés à l'intérieur. Nous devons impérativement corriger ce typage informatique, car l'ordinateur ne pourra pas faire de mathématiques ou de statistiques sur du texte.

### 3. Uniformisation des Formats de Dates

Le dataset Automobile ne contient pas de colonne de dates. Cette section est consacrée au traitement de la seule valeur manquante identifiée lors de l'audit : la colonne **horsepower** (6 valeurs NaN). On impute ces valeurs par la médiane de la distribution.

In [4]:
# Affichage des 6 lignes où horsepower est NaN
print("=== Lignes avec horsepower manquant ===")
print(df_raw[df_raw['horsepower'].isnull()])

# Calcul et affichage de la médiane
hp_median = df_raw['horsepower'].median()
print(f"\nMédiane de horsepower : {hp_median}")

# Imputation par la médiane
df_raw['horsepower'] = df_raw['horsepower'].fillna(hp_median)

# Vérification : aucune valeur manquante restante
print("\n=== Valeurs manquantes après imputation ===")
print(df_raw.isnull().sum())

=== Lignes avec horsepower manquant ===
                     name   mpg  cylinders  displacement  horsepower  weight  \
32             ford pinto  25.0          4          98.0         NaN    2046   
126         ford maverick  21.0          6         200.0         NaN    2875   
330  renault lecar deluxe  40.9          4          85.0         NaN    1835   
336    ford mustang cobra  23.6          4         140.0         NaN    2905   
354           renault 18i  34.5          4         100.0         NaN    2320   
374        amc concord dl  23.0          4         151.0         NaN    3035   

     acceleration  model_year  origin  
32           19.0          71     usa  
126          17.0          74     usa  
330          17.3          80  europe  
336          14.3          80     usa  
354          15.8          81  europe  
374          20.5          82     usa  

Médiane de horsepower : 93.5

=== Valeurs manquantes après imputation ===
name            0
mpg             0
cylinder

**Interprétation de l'état des lieux :** Contrairement à beaucoup de jeux de données bruts, celui-ci est déjà très bien structuré dès le départ :
* **Le Typage (dtypes) est correct :** Toutes nos colonnes quantitatives (`mpg`, `horsepower`, `weight`, etc.) sont bien reconnues comme des nombres (`float64` ou `int64`). Il n'y a pas de problème de caractères cachés qui forcent le type en `object`.
* **Valeurs manquantes (NaN) :** Le dataset est presque complet. Seule la colonne `horsepower` (puissance en chevaux) contient des valeurs manquantes (exactement 6). 
* **Doublons :** Aucun doublon n'a été détecté.

**Plan d'action pour le Data Wrangling :**
Pour finaliser la préparation de nos données et les rendre parfaitement exploitables par nos futurs modèles de Machine Learning, nous allons exécuter les étapes suivantes :

1. **Imputation des NaN :** Remplacement des 6 valeurs manquantes de `horsepower` par la moyenne de la colonne.
2. **Suppression de la colonne `name` :** Cette variable nominale présente une cardinalité trop élevée (305 valeurs uniques). <br>
   Intégrer une telle variable textuelle brute risque de saturer le modèle et de provoquer du surapprentissage (*overfitting*).
3. **Encodage de la variable `origin` :** Les algorithmes ne comprenant que les chiffres, nous utilisons `LabelEncoder` pour transformer cette variable catégorielle en valeurs numériques exploitables.
4. **Détection des valeurs aberrantes (Méthode IQR) :** Nous appliquons la méthode de l'Écart Interquartile (IQR) sur nos 6 colonnes numériques afin d'identifier et d'isoler les *outliers* qui pourraient fausser les calculs statistiques et nos futures prédictions.

### 4. Identification et Filtrage des Valeurs Aberrantes (Outliers)

Suppression de la colonne `name` (305 valeurs uniques, purement nominale), encodage de la variable catégorielle `origin` avec `LabelEncoder`, puis détection des outliers par la méthode IQR sur les 6 colonnes numériques.

In [5]:
# Suppression de la colonne 'name' (identifiant nominal sans valeur prédictive)
df_clean = df_raw.drop(columns=['name'])
print(f"Shape après suppression de 'name' : {df_clean.shape}")

# Encodage de la variable catégorielle 'origin' avec LabelEncoder
le = LabelEncoder()
df_clean['origin'] = le.fit_transform(df_clean['origin'])

print("\n=== Mapping d'encodage de 'origin' ===")
for label, code in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {label} → {code}")

Shape après suppression de 'name' : (398, 8)

=== Mapping d'encodage de 'origin' ===
  europe → 0
  japan → 1
  usa → 2


In [14]:


# Vos colonnes numériques
numeric_cols = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']

# 1. Calcul des outliers
outlier_counts = {}
for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    outlier_counts[col] = n_outliers

print("=== Nombre d'outliers détectés (méthode IQR) ===")
for col, count in outlier_counts.items():
    print(f"  {col} : {count} outlier(s)")

# 2. Création de la grille dynamique 2x3 avec Plotly
fig = make_subplots(
    rows=2, cols=3, 
    subplot_titles=[f"{col}<br>({outlier_counts[col]} outlier(s))" for col in numeric_cols],
    vertical_spacing=0.15
)

# 3. Ajout des Boxplots interactifs (simplifiés pour utiliser le template)
for i, col in enumerate(numeric_cols):
    row = (i // 3) + 1
    col_pos = (i % 3) + 1
    
    fig.add_trace(
        go.Box(
            y=df_clean[col].dropna(), 
            name=col, 
            boxpoints='outliers', # Montre uniquement les outliers
            marker=dict(size=6)   # On définit juste la taille, le template gère la couleur !
        ), 
        row=row, col=col_pos
    )

# 4. Mise en page globale avec le NOUVEAU TEMPLATE
fig.update_layout(
    title_text="<b>Détection des Outliers par Méthode IQR — Dataset Automobile</b>",
    title_font_size=16,
    height=700,
    showlegend=False,
    template="plotly_white"  # <--- C'EST ICI QUE LA MAGIE OPÈRE
)

# Affichage du graphique interactif
fig.show()

print("\n=== Statistiques descriptives après encodage ===")
display(df_clean.describe())

=== Nombre d'outliers détectés (méthode IQR) ===
  mpg : 1 outlier(s)
  cylinders : 0 outlier(s)
  displacement : 0 outlier(s)
  horsepower : 11 outlier(s)
  weight : 0 outlier(s)
  acceleration : 7 outlier(s)



=== Statistiques descriptives après encodage ===


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
count,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000,398.000000
mean,23.514573,5.454774,193.425879,104.304020,2970.424623,15.568090,76.010050,1.449749
std,7.815984,1.701004,104.269838,38.222625,846.841774,2.757689,3.697627,0.775076
min,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,70.000000,0.000000
25%,17.500000,4.000000,104.250000,76.000000,2223.750000,13.825000,73.000000,1.000000
50%,23.000000,4.000000,148.500000,93.500000,2803.500000,15.500000,76.000000,2.000000
75%,29.000000,8.000000,262.000000,125.000000,3608.000000,17.175000,79.000000,2.000000
max,46.600000,8.000000,455.000000,230.000000,5140.000000,24.800000,82.000000,2.000000


**Ce que nous apprennent ces 6 diagrammes :**

* **`horsepower` (Puissance) et `acceleration` :** On observe clairement quelques points orange isolés au-dessus ou en dessous des "moustaches" (les lignes noires extrêmes). Ce sont nos *outliers*. Ils représentent probablement des voitures de sport très puissantes ou des véhicules très lents.
* **`mpg`, `weight` (Poids) et `displacement` :** La répartition est propre. Presque tous les points sont bien contenus à l'intérieur des limites de la boîte. Il n'y a pas de valeurs aberrantes problématiques ici.
* **`cylinders` (Cylindres) :** Le graphique a une forme très plate et "rayée". C'est tout à fait normal ! Le nombre de cylindres n'est pas une valeur continue (on ne peut pas avoir 4.2 cylindres), ce sont des catégories fixes (3, 4, 6, 8). 

**Conclusion :** Ces visualisations nous prouvent que les valeurs extrêmes détectées ne sont pas des "erreurs de frappe" (comme une voiture de 10 000 kg), mais correspondent à des modèles de véhicules rares mais réels. Il est donc justifié de les conserver pour ne pas amputer notre futur modèle prédictif.

### 5. Imputation des valeurs manquantes

Validation finale de l'absence de valeurs manquantes et confirmation des dimensions du dataset nettoyé.

In [ ]:
print("=== Valeurs manquantes restantes ===")
print(df_clean.isnull().sum())

print(f"\nDimensions finales du dataset nettoyé : {df_clean.shape}")

=== Valeurs manquantes restantes ===
mpg             0
cylinders       0
displacement    0
horsepower      0
weight          0
acceleration    0
model_year      0
origin          0
dtype: int64

Dimensions finales du dataset nettoyé : (398, 8)


### 6. Sauvegarde des données propres

Enregistrement du dataset nettoyé dans le répertoire `data/processed/`.

In [ ]:
processed_path = '../data/processed/cleaned_data_sample.csv'
os.makedirs(os.path.dirname(processed_path), exist_ok=True)
df_clean.to_csv(processed_path, index=False)

print(f"Données propres sauvegardées dans : {processed_path}")
print(f"Shape finale : {df_clean.shape}")
print("Wrangling terminé avec succès !")

Données propres sauvegardées dans : ../data/processed/cleaned_data_sample.csv
Shape finale : (398, 8)
Wrangling terminé avec succès !
